# S01 - ML foundations and Python stack
Draft pending professor review. Issue #13. Proposed mode: online. Block: B01.

## Approved outcome wording (from the curriculum)
- LO1: Explain and compare supervised, unsupervised, and reinforcement learning paradigms, identifying appropriate algorithms for given problem types.

## What you will do
Classify problems by available signal, decision and feedback; run a transparent supervised baseline to inspect the workflow; then design an independent sensitivity study. This notebook supports LO1 through explanation and comparison. Its tiny predictor is not a regression methods lesson or a claim about real-world performance. All activities are formative.
Prerequisites: basic Python expressions, lists, dictionaries and a sample mean. If these are unfamiliar, trace the rows with a partner and use the guided cell before the independent study. Notation: x is one dimensionless feature, y is a dimensionless response, T is training IDs, V is validation IDs, and n and m are their sizes.
Keep completed answers in your private copy. Read the exercise sheet and lecture source for the three-paradigm comparison; no private account details or completed work belong in this public repository.


## Setup and recovery
Open this .ipynb using Colab's File > Upload notebook, then save your own copy.
A Google account and internet connection are needed for hosted execution. Choose a Python 3 CPU runtime; no accelerator, paid tier, secrets, Drive mount or dataset login is needed.
Run all cells from the top in a fresh runtime. After changes, restart the runtime and run all again. If state is confusing, disconnect and delete the runtime, reconnect and rerun.
Use Python 3.10–3.13 on CPU. The dependency cell checks NumPy 2.1.3 and pandas 2.2.3, installs those exact versions only if needed, then records the versions actually imported. Installation needs internet; run it before any NumPy or pandas import. If the runtime had already loaded different versions, restart the runtime and Run all again. The remaining code uses only the standard library plus these two packages.
Keep your completed work private; do not submit it to the public repository. Download your copy before leaving. Runtime state is temporary.
Offline fallback: use the downloaded notebook in an already provisioned local Jupyter environment, or read the Markdown and trace the small table on paper. Arrange later execution if compute is unavailable; a paper trace does not demonstrate implementation.


In [ ]:
import sys
import json
import hashlib
from fractions import Fraction

assert (3, 10) <= sys.version_info[:2] <= (3, 13), "Use Python 3.10–3.13"
SEED = 42
print(json.dumps({"python": sys.version.split()[0], "seed": SEED, "hardware": "CPU"}, sort_keys=True))


In [ ]:
from importlib import metadata as package_metadata
import subprocess

REQUIRED = {"numpy": "2.1.3", "pandas": "2.2.3"}
installed = {}
for package in REQUIRED:
    try:
        installed[package] = package_metadata.version(package)
    except package_metadata.PackageNotFoundError:
        installed[package] = None
if installed != REQUIRED:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *[f"{name}=={version}" for name, version in REQUIRED.items()]])
import numpy as np
import pandas as pd
assert {"numpy": np.__version__, "pandas": pd.__version__} == REQUIRED, "Restart the runtime and Run all again"
print(json.dumps({"numpy": np.__version__, "pandas": pd.__version__}, sort_keys=True))


## Scientific Python stack: roles before imports
Python controls the program and records its environment. NumPy represents homogeneous numerical arrays; pandas represents labelled tables; Matplotlib makes visual checks; scikit-learn supplies estimators, preprocessing pipelines and evaluation tools. Colab hosts a Python runtime. A library cannot choose a valid target, split or error cost for you.
This first lesson converts one original list of dictionaries into a NumPy array and a pandas DataFrame. Compare their shapes, column selection and descriptive statistics. The exact-fraction baseline below keeps the ML workflow transparent. S02 owns cleaning, encoding, scaling and fuller EDA with pandas/scikit-learn under LO2.


## Data provenance and assumptions
This original synthetic teaching dataset, `m5-synthetic-v1`, is generated below; it has no external source, copied records or third-party dataset licence requirement. It is not a sample of people. The generator and canonical JSON SHA-256 identify the data; retain both when reporting an experiment.
Each row has an integer row_id, a dimensionless feature x and dimensionless response y. The arbitrary relation is y = 3x + 2 + e, with a small deterministic perturbation e. This is a designed illustration, not evidence about a population.
The seed selects a reproducible partition by sorting SHA-256 keys. It is not a scientific random sampling mechanism. Independent observations are assumed for this illustration; temporal or grouped observations need a different validation design.


In [ ]:
DATA_VERSION = "m5-synthetic-v1"
rows = [{"row_id": i, "x": i - 12, "y": 3 * (i - 12) + 2 + ((i * 7) % 5 - 2)} for i in range(24)]
payload = json.dumps(rows, sort_keys=True, separators=(",", ":")).encode("utf-8")
data_sha256 = hashlib.sha256(payload).hexdigest()
ordered_ids = sorted(range(len(rows)), key=lambda i: hashlib.sha256(f"{SEED}:{i}".encode("ascii")).hexdigest())
train_ids, validation_ids = ordered_ids[:18], ordered_ids[18:]
train = [rows[i] for i in train_ids]
validation = [rows[i] for i in validation_ids]
assert set(train_ids).isdisjoint(validation_ids)
assert sorted(train_ids + validation_ids) == list(range(24))
assert len(train) == 18 and len(validation) == 6
print(json.dumps({"data_version": DATA_VERSION, "sha256": data_sha256, "train_ids": train_ids, "validation_ids": validation_ids}, sort_keys=True))


In [ ]:
assert all(set(row) == {"row_id", "x", "y"} for row in rows)
assert all(type(row["x"]) is int and type(row["y"]) is int for row in rows)
print(json.dumps({"shape": [len(rows), 3], "columns": ["row_id", "x", "y"], "first_three_rows": rows[:3]}, sort_keys=True))


## NumPy: an array has a shape and positional columns
Make a 24 × 2 integer array from the existing x and y values. The first axis counts rows; the second counts measured columns. The row identifier is deliberately excluded. Selecting all rows and column 0 returns the x values. A mean describes these invented rows; it does not estimate a real population without a sampling argument.


In [ ]:
measurements = np.asarray([[row["x"], row["y"]] for row in rows], dtype=np.int64)
assert measurements.shape == (24, 2)
x_array = measurements[:, 0]
assert x_array.shape == (24,)
print(json.dumps({"array_shape": list(measurements.shape), "first_three_pairs": measurements[:3].tolist(), "first_three_x": x_array[:3].tolist(), "x_mean": float(np.mean(x_array))}, sort_keys=True))


## pandas: labelled columns and descriptive statistics
Build a DataFrame from the same rows. Its 24 × 3 shape includes row_id, x and y; a DataFrame label records meaning that the NumPy column position alone does not. Select x and y by name and inspect count, mean, minimum and maximum. These summaries describe this constructed table, not model performance or group fairness.


In [ ]:
frame = pd.DataFrame.from_records(rows, columns=["row_id", "x", "y"])
assert frame.shape == (24, 3)
selected = frame[["x", "y"]]
assert selected.shape == measurements.shape
assert frame["x"].to_numpy().tolist() == x_array.tolist()
summary = selected.describe().loc[["count", "mean", "min", "max"]]
print("First three labelled rows\n" + frame.head(3).to_string(index=False))
print("Selected-column summary\n" + summary.to_string())
assert float(frame["x"].mean()) == float(np.mean(x_array))


Guided check: Why does the array have two columns while the DataFrame has three? Which expression selects x by position, and which selects it by name? Compare the two x means. State why agreement is a representation check, not evidence that a learning algorithm is accurate. Then return to the LO1 question: which additional signal would make this table a supervised learning task?


## Worked reproducibility demonstration
This public worked example is a reusable workflow illustration, not an independent exercise answer or a complete session lab.
For n training responses, the constant baseline is c = (sum of y_i) / n. It minimises the training sum of squared errors among constant predictors: differentiating gives 2nc - 2 sum(y_i) = 0; the second derivative is 2n > 0.
For m validation cases, MSE = (sum of (y_i - c)^2) / m, in squared response units. Lower is better under squared-error loss, which penalises large residuals strongly. This metric is sensitive to outliers; the constant model ignores x.
Split before any fitted preprocessing or estimation. Here there is no preprocessing; the baseline uses training responses only. Validation is for investigation, not repeated claims of final generalisation. A later modelling lab needs fold-local preprocessing, a justified cross-validation design and a reserved final evaluation set where appropriate.
Exact rational arithmetic makes this small demonstration independent of floating-point libraries. A single fixed split demonstrates repeatability, not uncertainty quantification or robustness.


In [ ]:
baseline = Fraction(sum(row["y"] for row in train), len(train))
validation_mse = sum((Fraction(row["y"]) - baseline) ** 2 for row in validation) / len(validation)
assert validation_mse >= 0
print(json.dumps({"baseline": str(baseline), "validation_mse": str(validation_mse)}, sort_keys=True))


## Guided activity (formative proposal)
Before running the demonstration, predict what the constant model can and cannot learn. Read the schema preview as a three-column table: row_id is an identifier, x a feature and y the response. Explain why row_id is not automatically a defensible predictor. Trace which rows contribute to the estimate and which only contribute to its validation score.
Restart and run all twice; compare Python version, data hash, split IDs and exact rational results. Discuss why matching results do not establish model quality. Explain why fitting a future transformation on all rows would contaminate validation. Online: compare reasoning verbally or in text. Campus: rotate driver and reviewer, then each record an explanation privately.


## Independent practice (formative proposal)
STUDENT TODO: State a hypothesis about the sensitivity of this baseline to partition choice. Plan an experiment before coding; choose a finite seed list and record each split and score. Keep the original run as your reference. Do not tune a model against a final holdout.
STUDENT TODO: Implement your experiment in the next cell. Produce a nonempty list of records with integer seed and nonnegative finite numeric mse. The format check is not a correctness test or grade. Explain what variation across partitions can and cannot tell you; consider dependence between splits.
STUDENT TODO: For one invented scenario in exercises.md, defend a supervised, unsupervised or reinforcement formulation, then change one goal or feedback assumption and defend a second formulation. Name plausible algorithm families and evidence that would distinguish them.
STUDENT TODO: Propose one methodological alternative to the constant baseline and justify its assumptions and cost without fitting it here. Explain why synthetic performance cannot establish fairness or effectiveness on real data. Retain your answers privately.


In [ ]:
# STUDENT TODO: Implement your planned experiment; leave None until you attempt it.
experiment_results = None


In [ ]:
import math

if experiment_results is None:
    print("NOT ATTEMPTED: independent practice; scaffold execution is not completion.")
else:
    assert isinstance(experiment_results, list) and experiment_results, "Return a nonempty list"
    for record in experiment_results:
        assert isinstance(record, dict) and set(record) == {"seed", "mse"}, "Use seed and mse fields"
        assert type(record["seed"]) is int, "seed must be an integer"
        assert type(record["mse"]) in (int, float) and math.isfinite(record["mse"]) and record["mse"] >= 0, "mse must be finite and nonnegative"
    print("FORMAT CHECK PASSED: methodological correctness still needs review.")


## Reflection and evidence
STUDENT TODO: Record your hypothesis, environment, data version/hash, split design, results, interpretation and limitations in your private copy. Distinguish reproducibility from statistical reliability and societal fairness from bias-variance.
STUDENT TODO: Explain which task is supervised learning here, and how an unsupervised or reinforcement learning formulation would require different evidence. Do not claim this small demonstration fulfils all session outcomes.
This notebook contains original explanations and synthetic data, with no copied assets or external readings embedded. Verified link-only S01 readings are in references.md.
